# EDA foundations — the checklist you will reuse

Data Science & AI. Module 2 Part 1, slides 3-51.

**This notebook is not a lecture.** It is the reference you keep.

Three reasons it exists:

1. If you missed the Saturday session, this is the same ground, in the
   same order, self-contained. Nothing here assumes you were in the room.
2. It is the **EDA checklist** — the thing you open on day one of every
   future project, including Mini Project 1, and work down.
3. Monday's session opens by speed-running sections 1-4 on this exact
   dataset, so an hour here on Sunday is an hour well spent.

Work through it at your own pace. Everything runs.

## How to use this file

**Make your own copy first** — File → Make a Copy, or `cp` it into your
`my-work/` folder. Then **Restart & Run All** once, so you know the
baseline works before you start changing things.

There are four **deliberate errors** in here. Two raise a traceback and
two fail silently, which is the more dangerous kind. Each is announced,
and each is followed by the fix.

---

## 1. Where data comes from, and what it looks like  *(slides 4, 5)*

Before any of the tooling, a habit: **know your source**. It determines
what can go wrong, and "what can go wrong" is most of this notebook.

| Source *(slide 4)* | What it means for you |
|---|---|
| **Databases** | Typed, constrained, usually the cleanest thing you will get. Someone already decided what a valid row is. |
| **Transaction systems** | Built to record events fast, not to be analysed. Expect codes, not labels. |
| **Distributed file systems** (HDFS, S3) | Too big for one machine. You will be sampling. |
| **APIs** | JSON, paginated, rate-limited, and the shape can change without warning. Module 3. |
| **Scanned documents** | OCR output. Assume errors *in the values themselves*. |
| **Websites** (scraping) | No contract at all. A layout change breaks you silently. Module 8. |
| **Subscribed feeds** | Someone else's schema, someone else's outage. |
| **Multimedia** | Images, audio, video — features have to be extracted before anything else. |

And the shapes it arrives in *(slide 5)*: database tables, reports and
extracts, spreadsheets, structured and semi-structured files (CSV, JSON,
XML), streams, encoded files, bitmaps.

The gap between those two lists is where the job lives. Nobody hands you
a clean DataFrame.

## 2. What EDA is, and where it fits  *(slides 6, 7, 8)*

The deck's definition is precise, and the two boundaries in it matter
more than the middle:

> Everything we do with a candidate dataset **after** it has been rendered
> essentially usable, and **before** we start developing analytics and
> models — to determine whether it will make a useful **proxy** for the
> phenomenon we actually care about.

Three things worth pulling out.

**"Candidate."** The dataset is on trial. EDA can, and sometimes should,
end with "this data cannot answer that question". That is a successful
EDA, delivered early and cheaply.

**"Proxy."** You never measure the phenomenon. You measure a trace of it.
We are about to study *bike hires*, which is a proxy for *demand for
bikes* — and it is a poor one at 8am when every dock is empty, because a
person who wanted a bike and found none leaves no row in the file. Ask
what your data cannot see, every time.

**"Before models."** Every hour here saves a day later. A model trained
on a column you did not understand will train perfectly and be wrong.

### Making a dataset usable  *(slide 7)*

Three words the deck uses, often muddled in practice:

- **Wrangling** — getting hold of it and into a workable shape. Loading,
  joining, reshaping.
- **Profiling and cleaning** — what this notebook is about. Measuring what
  you have, then fixing what is broken.
- **Munging** — transforming values into what the analysis needs.
  Deriving, encoding, aggregating.

### Where EDA sits  *(slide 8)*

    Define  ->  Prepare  ->  Analyse  ->  Deliver
                   \____________/
                          EDA

**The deck's most important line about this diagram is that the process is
never linear.** You will be three days into Analyse when a chart makes no
sense, and the cause will be a parsing decision from Prepare. Going back
is not failure. It is the method.

---

## 3. The clean-and-profile loop  *(slides 9, 10, 11)*

Two definitions that get used interchangeably and should not be
*(slide 10)*:

- **Data profiling** — *measuring*. What is in here, what types, what
  ranges, how much is missing, how is it distributed. Profiling changes
  nothing.
- **Data cleaning** — *changing*. Fixing, converting, filling, dropping.

You profile to decide what to clean, then profile again to see what your
cleaning did. Slide 11 draws it as a loop, and it is a loop:

    load raw data -> fix loading errors -> parse and convert
          ^                                      |
          |                                      v
    detect & fix invalid <- detect & fix missing <- summarise

**Never clean without profiling afterwards.** Filling 900 missing values
with the mean changes the standard deviation of that column, and if you
do not look, you will not know.

Let us do a full lap.

### 3.1 Load raw data  *(slide 12)*

Our source is a flat file. Here is the honest version of loading one —
try locally, fall back to the internet:

In [ ]:
# ==========================================================
# Setup. Run this once, then carry on.
# ==========================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

UCI = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
LOCAL_FILE = "data/bikeshare-hour.csv"


def load_bikes():
    """Local copy if it is beside us. Otherwise download it once and save it there."""
    try:
        return pd.read_csv(LOCAL_FILE)
    except FileNotFoundError:
        pass

    import io
    import urllib.request
    import zipfile

    print("no local copy — downloading from UCI ...")
    with urllib.request.urlopen(UCI, timeout=60) as response:
        archive = zipfile.ZipFile(io.BytesIO(response.read()))

    # The zip holds three files, so we have to name the one we want.
    with archive.open("hour.csv") as csv_file:
        data = pd.read_csv(csv_file)

    data.to_csv(LOCAL_FILE, index=False)
    print(f"saved to {LOCAL_FILE} — next run will load it straight from disk")
    return data


bikes = load_bikes()

pd.set_option("display.max_columns", 20)
print(bikes.shape)

**The dataset.** Every hour Capital Bikeshare operated in Washington DC
across 2011 and 2012, plus the weather at the time and how many bikes
went out.

**One warning before you open the lab file.** IOD's lab uses a different
cut of the same data — the Kaggle "Bike Sharing Demand" version, 10,886
rows, columns named `datetime`, `weather`, `humidity`, `count`. This is
the parent source from UCI: both full years, columns `dteday` + `hr`,
`weathersit`, `hum`, `cnt`. Same bikes, different packaging. If the names
do not match, nothing is broken.

### 3.2 Fix loading errors  *(slide 13)*

That file loaded cleanly. Most will not.

Slide 13 lists what goes wrong: missing delimiters, unexpected delimiters,
illegal characters, missing control characters. All four are the same
underlying problem — **the file's rules and the reader's assumptions
disagree**, and `read_csv` will not tell you.

Here is a file like the ones that will actually land in your inbox. We
write it out so you can see exactly what is in it:

In [ ]:
messy_text = """member_id;joined;home_station;annual_fee;trips_2012
1001;2011-03-14;Columbus Circle / Union Station;85.00;312
1002;2011-07-02;14th & V St NW;85.00;1 204
1003;;Smithsonian;85.00;7
1004;2012-01-19;Adams Mill & Columbia Rd NW;N/A;58
1005;30/04/2012;Eastern Market Metro;75.00;441
1006;2012-06-11;15th & P St NW;85.00 ;
1007;2012-09-05;"Lincoln Memorial; Reflecting Pool";85.00;96
"""

with open("members.csv", "w") as f:
    f.write(messy_text)

print(messy_text)

> **Predict first.** We are about to `read_csv` that file with default settings. How many columns will pandas report?
>
> Write down your answer before you run the cell.

In [ ]:
# DELIBERATE ERROR 1 of 4 — this one does NOT raise. That is the problem.
members_bad = pd.read_csv("members.csv")

print("columns pandas found:", len(members_bad.columns))
print(members_bad.columns.tolist())
members_bad.head(3)

**One column.** No error, no warning, no traceback. A DataFrame came
back, `.head()` printed something that looks vaguely like a table, and
every value you own is now one long string.

The file is **semicolon-delimited** — normal across most of Europe, where
the comma is the decimal separator. `read_csv` assumed a comma, found
none, and concluded that each line is a single field.

This is the single most common loading failure and it is silent. Which
gives the first rule:

> **After every load, check the shape.** If the column count is 1, or is
> not what the file promised, stop.

The fix is one argument:

In [ ]:
members = pd.read_csv("members.csv", sep=";")
print(messy_text)
print("shape:", members.shape)
members

Now look at what survived, because the delimiter was only the first
problem. Every one of these is on slide 13's list:

- **`1007`'s station contains a semicolon** — *"Lincoln Memorial;
  Reflecting Pool"*. It survived only because the file quoted it. An
  **unquoted** delimiter inside a value shifts every column to its right
  on that row — and that one usually *does* raise, because the row then
  has more fields than the header. A rare kindness.
- **`1002`'s trip count is `1 204`** — a space as a thousands separator,
  normal in French and Scandinavian exports. The column is now text.
- **`1004`'s fee is `N/A`** — a string sitting in a numeric column.
- **`1006`'s fee is `"85.00 "`** with a trailing space, and its trip
  count is empty.
- **`1003`'s join date is empty**; **`1005`'s is `30/04/2012`** while
  everyone else's is ISO.

The file loads. Nothing raises. And almost nothing in it is usable yet.

### 3.3 Parse and convert  *(slides 14, 15)*

Slide 14's list: date strings to dates, time strings to times, datetime
strings to datetimes, string to int, string to float, proprietary
formats.

Every one is the same operation — **give a column its real type** — and
every one can fail on a single bad row.

In [ ]:
print(members.dtypes)

Read that carefully, because there is a surprise in it.

`annual_fee` came back as **`float64`** — already numeric, despite
containing the string `N/A` and the value `"85.00 "` with a trailing
space. `read_csv` dealt with both without being asked.

The trailing space is simple: the parser strips whitespace. The `N/A` is
not simple, and it is worth knowing exactly what happened:

In [ ]:
from pandas.io.parsers.readers import STR_NA_VALUES

print(sorted(STR_NA_VALUES))

**`read_csv` treats every one of those strings as missing, by default.**
`N/A` is on the list, so member 1004's fee became `NaN` during loading
and the column stayed numeric.

Usually that is exactly what you want. Occasionally it is a disaster,
and here is the classic:

> **Predict first.** A file of country codes has one row for Namibia, whose ISO code is `NA`. What does pandas load?
>
> Write down your answer before you run the cell.

In [ ]:
with open("countries.csv", "w") as f:
    f.write("country;code\nNamibia;NA\nCanada;CA\nNorway;NO\n")

display(pd.read_csv("countries.csv", sep=";"))

**Namibia has vanished.** Its code is a real value that happens to spell
one of pandas' missing markers, and nothing warned you. The same bug
bites gene name `NA`, chemical symbol `NA`, and any survey where "NA"
means "North America".

The fix is to say so explicitly:

In [ ]:
safe = pd.read_csv("countries.csv", sep=";", keep_default_na=False)
print("with keep_default_na=False:", safe["code"].tolist())

> **Know what your loader silently converts.** `keep_default_na=False`
> turns the whole list off; `na_values=[...]` sets your own.

Now the column that really did stay text — `trips_2012`:

In [ ]:
members

> **Predict first.** `trips_2012` holds `312`, `1 204`, `7`, `58`, `441`, a blank, `96`. What does `pd.to_numeric` do with it?
>
> Write down your answer before you run the cell.

In [ ]:
members

In [ ]:
# DELIBERATE ERROR 2 of 4 — this one raises. Read the LAST line first.
pd.to_numeric(members["trips_2012"])

`ValueError: Unable to parse string "1 204" at position 1`

Read it bottom-up. It even gives you the row. One string in seven stopped
the whole conversion — which is *correct behaviour*. Silence would be
worse, as Namibia just demonstrated.

Slide 15 asks the right question: **what do you do when conversions
fail?** It offers three answers, and you will use all three.

**1. Catch it, row by row.** The deck's own helper:

In [ ]:
def try_parse_int(s,  val=None): #base=10,
    """Slide 15's function: return None instead of raising."""
    try:
        return int(s) #, base
    except ValueError:
        return val


print(try_parse_int("312"))
print(try_parse_int("1 204"))     # -> None, and the loop keeps going
print(try_parse_int(321))

**2. Use a transformation that tolerates failure.** Pandas has this
built in, and it is what you will actually reach for — `errors="coerce"`
turns anything unparseable into `NaN` instead of raising:

In [ ]:
coerced = pd.to_numeric(members["trips_2012"], errors="coerce")

print(coerced.tolist())
print()
print("became NaN:", coerced.isna().sum(), "of", len(coerced))

Look at what that cost. **Two** `NaN`s, and they are not the same kind of
thing:

- Member 1006's trip count was genuinely blank. `NaN` is the truth.
- Member 1002's was `1 204` — **a real value, destroyed by a space**.

`coerce` cannot tell those apart, and neither can anyone reading the
result later. Clean what you can *explain* first, and coerce only what is
left:

In [ ]:
trips = pd.to_numeric(members["trips_2012"].str.replace(" ", "", regex=False),
                      errors="coerce")

print(trips.tolist())
print("still missing:", trips.isna().sum(), "(member 1006, genuinely blank)")

1,204 recovered. The general rule: **`coerce` last, not first.**

**3. Document the failures.** The step everyone skips, and slide 15 is
right to list it. `coerce` is a decision to throw information away, so
say what you threw:

In [ ]:
lost = members.loc[trips.isna(), ["member_id", "trips_2012"]]
print("rows where trips_2012 could not be parsed:")
print(lost)

One row, blank, member 1006. Now a known unknown rather than a silent
zero — and if it had been ten thousand rows you would have found out
before building anything on top of it.

Now the dates:

> **Predict first.** `joined` holds six ISO dates, one blank, and one written `30/04/2012`. What will `pd.to_datetime` do with the odd one out?
>
> Write down your answer before you run the cell.

In [ ]:
members

In [ ]:
print(type(members["joined"]))
joined = pd.to_datetime(members["joined"], format="mixed", dayfirst=True)
print(joined)

`format="mixed"` lets pandas work row by row, and `dayfirst=True` tells
it that `30/04/2012` is 30 April, not an invalid 4 October.

**Be careful here.** `01/02/2012` is ambiguous — 1 February to most of
the world, 2 January in the US. Pandas cannot know. If your file mixes
conventions, some dates will be silently wrong and *nothing will error*.
When you can, get the source to send you ISO (`YYYY-MM-DD`); when you
cannot, state the assumption in writing.

Why bother converting at all? Because a real date type knows things:

In [ ]:
# DELIBERATE ERROR 3 of 4 — raises. What can you do with a date-as-string?
members["joined"].dt.year

`AttributeError: Can only use .dt accessor with datetimelike values`

The `.dt` accessor is the whole reason to convert. On the converted
column it works:

In [ ]:
print(joined.dt.year.tolist())
print(joined.dt.day_name().tolist())

Now assemble a properly typed table. **Build a clean copy; do not mutate
the raw one** — when a number looks wrong three hours from now you will
want the original to compare against:

In [ ]:
clean = pd.DataFrame({
    "member_id": members["member_id"],
    "joined": joined,
    "home_station": members["home_station"].str.strip(),
    "annual_fee": members["annual_fee"],   # read_csv already made this float
    "trips_2012": trips,
})

print(clean.dtypes)
clean

> **Checkpoint.** `int64`, `datetime64`, `object`, `float64`, `float64`. Five columns, five correct types, and two `NaN`s that are honestly labelled instead of silently guessed.

---

## 4. Missing and invalid data  *(slides 16, 17)*

Now the decision everyone gets wrong. You have holes. What do you do?

Slide 17 gives four options, roughly in order of how much you should
like them:

| Option | When it is right | What it costs |
|---|---|---|
| **Replace with NaN** | Always, first. Make the hole visible and typed. | Nothing. Do this. |
| **Impute** | The column matters, the gaps are few, and you know why they are there. | You invent data. Variance shrinks. |
| **Drop rows** | Few rows, missing at random, plenty of data left. | Sample size, and possibly a whole subgroup. |
| **Drop columns** | The column is mostly empty or you cannot fix it. | Everything that column knew. |

The two questions on slide 19 are the ones to actually ask:

> **Can we afford to throw out rows with missing data?**
> **How will imputing affect the outcome?**

### 4.1 Find them first

In [ ]:
print(clean.isnull())
print(clean.isnull().sum())
print()
print("rows with any missing value:", clean.isnull().any(axis=1).sum(), "of", len(clean))

Three holes, in three different columns, on three different rows. So
dropping rows costs **3 of 7 members — 43% of the file, to fix three
cells**. 

### 4.2 The four options, actually run

In [ ]:
print("drop any row with a hole: ", clean.dropna().shape)
print("drop a row only if ALL are missing:", clean.dropna(how="all").shape)
print("drop columns with holes:  ", clean.dropna(axis=1).shape)

> **Predict first.** `clean.dropna()` just ran. How many rows does `clean` have now?
>
> Write down your answer before you run the cell.

In [ ]:
print("rows in clean:", len(clean))

**Still seven.** That is deliberate error 4 of 4, and it is the quietest
one in this notebook.

`dropna()` **returns a new DataFrame**. It does not change the original.
The cell above computed a five-row table, printed its shape, and threw it
away. Almost every pandas method behaves this way, and every one of us
has lost an afternoon to it.

Assign the result, or nothing happened:

In [ ]:
dropped = clean.dropna()          # assign it
print("dropped:", dropped.shape, " original untouched:", clean.shape)

### 4.3 Imputation, and what it costs  *(slide 16)*

Slide 16's menu: **mean**, **median**, **mode**, or a model-based guess.
Two more that matter for ordered data: **forward fill** (carry the last
value onward) and **interpolate** (draw a straight line across the gap).

Choosing between them is not a style question:

In [ ]:
fee_col = clean["annual_fee"]

print("mean  :", round(fee_col.mean(), 2))
print("median:", fee_col.median())
print("mode  :", fee_col.mode()[0])

All three are defensible for one missing fee. On a skewed column they
would not be — the mean of a heavily skewed column is dragged by its tail,
which is exactly the salary example from the statistics session.

Here is the cost, measured. Watch the standard deviation:

In [ ]:
filled = fee_col.fillna(fee_col.mean())

print("before: n = %d, sd = %.2f" % (fee_col.count(), fee_col.std()))
print("after : n = %d, sd = %.2f" % (filled.count(), filled.std()))

The spread **fell**. It had to: we added a point sitting exactly on the
mean, which contributes nothing to variance while still counting in the
denominator.

Mean-imputation always does this. Impute 30% of a column and you have
manufactured a dataset that looks more certain than the one you
collected — narrower confidence intervals, smaller p-values, more
confident model. All of it fake.

> **Impute if you must, then say so, and never report the spread of an
> imputed column as if you measured it.**

A safer habit, and one that costs nothing: **keep a flag**.

In [ ]:
clean["fee_was_missing"] = clean["annual_fee"].isna()
clean["annual_fee_filled"] = clean["annual_fee"].fillna(fee_col.median())

print(clean[["member_id", "annual_fee", "fee_was_missing",
             "annual_fee_filled"]])

Now every downstream step can choose. Nothing was lost, and the fact that
a value was invented travels with it.

### 4.4 Invalid values — the ones that are not missing

Missing data announces itself. **Invalid data hides as an ordinary
number**, which is why it is worse.

> **Predict first.** `hum` is humidity, normalised so 1.0 means 100%. Its minimum is 0.0. Is that a real reading?
>
> Write down your answer before you run the cell.

In [ ]:
bikes

In [ ]:
print("lowest real humidity: %.0f%%" % (bikes["hum"].min() * 100))
print("humidity readings of exactly 0:", (bikes["hum"] == 0).sum())

In [ ]:
display(bikes[bikes["dteday"] == "2011-03-10"])
print("dates:", sorted(bikes.loc[bikes["hum"] == 0, "dteday"].unique()))
print(len(bikes[bikes["dteday"] == "2011-03-10"]))

Air in Washington DC has never been perfectly dry. That is a sensor
reporting a failure as a number — 22 of them, and **every one on the same
day**.

That clustering is the finding. Twenty-two zeros scattered across two
years would suggest random dropouts. Twenty-two on one date is one
instrument, one fault, one morning. Different diagnosis, different fix.

`NaN` is the honest value, because "we do not know" is the truth:

In [ ]:
bikes["hum_clean"] = bikes["hum"].replace(0, np.nan)

print("now missing:", bikes["hum_clean"].isna().sum())
print("lowest real humidity: %.0f%%" % (bikes["hum_clean"].min() * 100))

> **Checkpoint.** Range checks catch what null checks cannot. For every numeric column, ask what its impossible values would look like — a negative age, a 0% humidity, a future birth date — and go looking for them.

---

## 5. Summarise, and assess quality  *(slides 18, 19, 20)*

Slide 20 is the practical one: the pandas calls that make up a first
profile. Here they are, in the order worth running them.

| Question | Call |
|---|---|
| What does it look like? | `df.head()`, `df.tail()` |
| How big, what types? | `df.shape`, `df.dtypes`, `df.info()` |
| What is missing? | `df.isnull().sum()` |
| Continuous ranges? | `df.describe()`, `df.min()`, `df.max()` |
| Discrete values? | `df["col"].value_counts()` |
| What moves with what? | `df.corr()` |

In [ ]:
bikes.info()

`info()` in one call: row count, every column, its non-null count, its
dtype, and the memory footprint. **Read the non-null column against the
row count** — that comparison is a missing-value audit for free.

In [ ]:
print(bikes[["temp", "hum", "windspeed", "casual", "registered", "cnt"]]
      .describe().round(3).T)

In [ ]:
# Check the earliest and latest dates in the dataset
print("First day of data:", bikes["dteday"].min())
print("Last day of data:", bikes["dteday"].max())

Now apply slide 19's quality questions to what you just read:

- **Accuracy and reliability (veracity).** `temp` maxes out at exactly
  1.000 and `hum` at exactly 1.000. Suspiciously round. The UCI
  documentation explains it: these columns are **normalised** — `temp`
  divided by 41, `atemp` by 50, `hum` by 100, `windspeed` by 67. So
  `temp = 0.5` is 20.5°C, not half a degree.

  **`describe()` looked entirely reasonable and meant nothing.** Read the
  data dictionary before you read the data.

- **Currency and relevance (value).** This is 2011-2012 data. Fine for
  learning weather-and-commuting patterns; useless for forecasting next
  week, because the system has tripled in size and e-bikes did not exist
  yet.

- **Missing and invalid.** `cnt` has a minimum of 1 — never 0. Either
  demand was never zero in two years, or **hours with no hires were not
  recorded**. That difference matters enormously and the file will not
  settle it.

Undo the normalisation so the rest of this notebook is in real units:

In [ ]:
bikes["temp_c"] = bikes["temp"] * 41
bikes["hum_pct"] = bikes["hum"] * 100
bikes["wind_kmh"] = bikes["windspeed"] * 67

print(bikes[["temp_c", "hum_pct", "wind_kmh"]].describe().round(1).T)

Believable numbers for Washington DC, which is the check that matters:
does this look like the world?

### Integrity checks belong in code

The documentation says `cnt` is casual plus registered. Do not trust it —
assert it, so it is re-checked every time the notebook runs:

In [ ]:
assert (bikes["cnt"] == bikes["casual"] + bikes["registered"]).all()
print("cnt = casual + registered on all", len(bikes), "rows")

### Automated profiling  *(slide 20)*

Slide 20 name-drops `pandas_profiling` and `pydqc`. The first has been
renamed — it is **`ydata-profiling`** now, and `pandas_profiling` no
longer installs. It generates a full HTML report in one line:

```python
# pip install ydata-profiling
from ydata_profiling import ProfileReport
ProfileReport(bikes, title="Bike share profile").to_notebook_iframe()
```

Genuinely useful for a first look at a wide, unfamiliar table. Two
honest caveats: it is slow on large files, and it will not tell you that
`temp` is divided by 41 — it cannot read documentation, and it does not
know what Washington DC feels like in July. **It profiles. It cannot
assess.**

In [ ]:
! pip install -q "setuptools<81" ydata-profiling
from ydata_profiling import ProfileReport
ProfileReport(bikes, title="Bike share profile").to_notebook_iframe()

In [ ]:
# 1. Create the report
report = ProfileReport(bikes, title="Bike share profile")

# 2. Save it directly to an HTML file
report.to_file("bike_share_report.html")

---

## 6. Outliers  *(slides 22-26)*

> **An outlier is an observation that is distant from other observations
> in the sample.**

Note what the definition does *not* say: that it is wrong. Slide 22 lists
where they come from — measurement inaccuracy, recording errors, unusual
system behaviour, external phenomena — and only the first two are
mistakes. The other two are **the most interesting rows in your file**.

### 6.1 One dimension: extreme value analysis  *(slide 23)*

Two standard rules. They will not agree.

In [ ]:
cnt = bikes["cnt"]

z = (cnt - cnt.mean()) / cnt.std()
by_z = (z.abs() > 3).sum()

q1, q3 = cnt.quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
by_iqr = ((cnt < low) | (cnt > high)).sum()

print("z-score rule (|z| > 3):     %5d rows" % by_z)
print("IQR rule (1.5 x IQR fence): %5d rows" % by_iqr)
print()
print("IQR fence runs from %.0f to %.0f" % (low, high))


Same column, same day, **two rules, two answers**, one of them twice the
other. Neither is wrong. They are answering different questions:

- The **z-score rule** uses the mean and standard deviation — and both are
  themselves dragged by the outliers you are hunting. On a skewed column
  it under-reports the long tail.
- The **IQR rule** uses quartiles, which extremes barely move. It is
  *robust*, and on skewed data it flags more.

Look at the fence: it runs from **−322 to 642**. A negative lower fence
on a count that cannot go below zero — that is the rule telling you the
column is skewed and it knows it.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(cnt, bins=50, color="lightsteelblue")

mu, sd = cnt.mean(), cnt.std()
ax.axvline(mu - 3*sd, color="crimson", ls="--", label="z = ±3")
ax.axvline(mu + 3*sd, color="crimson", ls="--")

ax.axvline(low,  color="darkgreen", ls=":", label="1.5 × IQR fence")
ax.axvline(high, color="darkgreen", ls=":")

ax.set_xlabel("cnt")
ax.set_ylabel("rows")
ax.legend()
plt.show()

In [ ]:
print("skewness: %.2f  (0 = symmetric, positive = long right tail)"
      % cnt.skew())
print("kurtosis: %.2f  (0 = normal-tailed, positive = heavy tails)"
      % cnt.kurt())

Those are slide 29's two shape numbers, and they justify the choice:
right-skewed, so **prefer the IQR rule here**.

Draw it, always. Slide 32's box plot is the picture of exactly that rule:

In [ ]:
plt.figure(figsize=(4, 5))
plt.boxplot(cnt)
plt.title("Hourly hires")
plt.ylabel("hires per hour")
plt.show()

The whiskers reach to the fence; the dots above them are the 505 rows.

**But look at how many dots there are.** When 3% of your data is
"outlying", the honest reading is not *"505 bad rows"* — it is
*"this distribution is skewed, and a rule designed for a bell curve is
mislabelling its tail"*. Deleting them would delete every busy hour of
the year.

### 6.2 The rule that is not obvious in one dimension  *(slide 26)*

Slide 26 makes the deepest point in this section, and it deserves a
demonstration.

Ask whether **300 hires in an hour** is unusual:

In [ ]:
print("300 hires, judged against the whole column:")
print("   z = %.2f — completely unremarkable"
      % ((300 - cnt.mean()) / cnt.std()))

Unremarkable. But *when* was it?

In [ ]:
for hour in [3, 17]:
    at_hour = bikes.loc[bikes["hr"] == hour, "cnt"]
    z_here = (300 - at_hour.mean()) / at_hour.std()
    print("at %02d:00 — mean %6.1f, sd %6.1f, so 300 has z = %6.2f"
          % (hour, at_hour.mean(), at_hour.std(), z_here))

**The same number, three verdicts.** Ordinary overall. Slightly *below*
average for 5pm. And at 3am, more than **twenty standard deviations** out
— a once-in-the-history-of-the-universe event.

That is slide 26 exactly: an outlier can hide inside a perfectly normal
one-dimensional range. `cnt` alone can never see it. `cnt` *given the
hour* can.

Do it properly — judge every row against its own hour:

In [ ]:
group = bikes.groupby("hr")["cnt"]
bikes["z_by_hour"] = (bikes["cnt"] - group.transform("mean")) / group.transform("std")

conditional = (bikes["z_by_hour"].abs() > 3).sum()
plain = (z.abs() > 3).sum()
hidden = ((bikes["z_by_hour"].abs() > 3) & (z.abs() <= 3)).sum()

print("flagged judging against the whole column:", plain)
print("flagged judging against the same hour:   ", conditional)
print("flagged ONLY by the conditional rule:    ", hidden)

**89 rows that the one-dimensional rule cannot see.** Every one is an
hour behaving unlike any other version of itself.

And this is where outlier detection stops being arithmetic:

In [ ]:
odd = bikes.nlargest(5, "z_by_hour")
print(odd[["dteday", "hr", "cnt", "z_by_hour"]].round(2).to_string(index=False))

Look at the top two dates rather than the numbers.

**7 November 2012, midnight** — 283 hires in the hour after midnight,
against a typical midnight of 54 and a typical 3am of 12. That is
election night in the United States; the result was called late on
6 November, and Washington DC did not go home.

**4 July 2012, 10pm** — Independence Day. Fireworks on the National Mall
end, and a hundred thousand people need to get home at once.

Neither is an error. Neither should be deleted. They are the deck's
*"external phenomena"*, and they are the most informative rows in a file
of 17,379.

> **Slide 25's rule: if you are unsure, analyse the data with and without
> them, and report both.** An outlier you cannot explain is a research
> question, not a mistake.

### 6.3 Several dimensions  *(slides 24, 26)*

What we just did by hand — condition on a second variable — generalises.
Slide 24 gives two families:

**Linear models.** Reduce the data to fewer dimensions, measure each
point's distance from the fitted plane, flag the far ones. Same machinery
as **PCA**, which arrives properly in Module 6.

**Proximity-based models.** Define a distance, then flag points that have
few neighbours. Cluster analysis, density methods like DBSCAN,
nearest-neighbour distance. Module 6 again.

You can see the idea with two columns and no library at all:

In [ ]:
day = bikes.groupby("dteday").agg(
    temp_c=("temp_c", "mean"),
    hires=("cnt", "sum"),
).reset_index()

plt.figure(figsize=(6.5, 4.5))
plt.scatter(day["temp_c"], day["hires"], s=10, alpha=0.5)
plt.xlabel("mean temperature (C)")
plt.ylabel("hires that day")
plt.title("Two columns. Which points look wrong TOGETHER?")
plt.show()

Warm days are busy days. Now find the days that break the deal — warm,
and empty anyway:

> **Predict first.** There is a cluster of points bottom-right: warm days with very few hires. Neither the temperature nor the hire count is extreme on its own. What kind of day is that?
>
> Write down your answer before you run the cell.

In [ ]:
warm = day[day["temp_c"] > 20]
suspects = warm.nsmallest(4, "hires")

print(suspects.round(1).to_string(index=False))
print()
weather = bikes.groupby("dteday")["weathersit"].max()
print("worst weather code on those days:")
print(weather.loc[suspects["dteday"]].to_string())

Warm days with almost no hires — and every one of them hit weather code
**3** at some point, which UCI defines as rain, snow or thunderstorms.
(The worst of them, 27 August 2011, is the day Hurricane Irene reached
Washington DC. Worth looking up rather than guessing.) Not outliers at
all once the third column is in the room. **A point that looks anomalous in two dimensions
often becomes ordinary in three.** That is the whole argument for
multivariate outlier detection, and for looking before you delete.

---

## 7. Exploring continuous data  *(slides 27-35)*

### 7.1 The four numbers that describe a distribution  *(slides 28, 29)*

- **Mean** — the balance point. **Variance / standard deviation** — the
  spread. Both from the statistics session.
- **Skewness** *(slide 29)* — which way the tail leans. Positive is a long
  right tail.
- **Kurtosis** — how heavy the tails are. Positive means extreme values
  are more common than a bell curve would predict, which is exactly when
  the z-score rule misleads you.

In [ ]:
for column in ["cnt", "temp_c", "hum_pct", "wind_kmh"]:
    values = bikes[column]
    print("%-9s mean %7.1f  sd %6.1f  skew %5.2f  kurt %6.2f"
          % (column, values.mean(), values.std(),
             values.skew(), values.kurt()))

Four columns, four different shapes, and you can predict each histogram
before drawing it. `cnt` leans hard right (skew 1.28) and has the
heaviest tails (kurtosis 1.42) — so extreme hours are commoner than a
bell curve would allow. `temp_c` is almost perfectly symmetric (skew
−0.01) with **negative** kurtosis, meaning flatter than a bell: a
four-season climate spends its time spread out, not clustered around a
mean.

### 7.2 Histogram  *(slide 31)*

Slide 31 makes a claim worth taking seriously: a histogram shows the
distribution **with no loss of information**. That is true of the data —
and false of the picture, because the bin count changes what you see:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, bins in zip(axes, [5, 30, 200]):
    ax.hist(bikes["cnt"], bins=bins)
    ax.set_title("bins = %d" % bins)
plt.tight_layout()
plt.show()

Same column, three times. Five bins hides the shape; two hundred bins is
noise; thirty is about right. **Change the bins before you believe a
histogram** — including someone else's.

In [ ]:
plt.figure(figsize=(8, 3))
plt.hist(bikes["temp_c"], bins=30)
plt.title("Temperature in Washington DC, 2011-2012")
plt.xlabel("degrees C")
#plt.ylabel("hours")
plt.show()

Broad and flat, with dips at both ends — that is a four-season climate
seen as a distribution, and a good example of a column whose histogram
should *not* be a bell.

### 7.3 Box plots and violins  *(slides 32, 33)*

A box plot is five numbers: minimum inside the fence, Q1, median, Q3,
maximum inside the fence — plus the dots beyond. Its strength is
**comparison**:

In [ ]:
plt.figure(figsize=(8, 4))
seasons = {1: "spring", 2: "summer", 3: "autumn", 4: "winter"}
data = [bikes.loc[bikes["season"] == s, "cnt"] for s in [1, 2, 3, 4]]

plt.boxplot(data, tick_labels=[seasons[s] for s in [1, 2, 3, 4]])
plt.ylabel("hires per hour")
plt.title("Hourly hires by season")
plt.show()

A **violin plot** *(slide 33)* keeps the comparison and adds the shape —
the deck's *"shows the sample distribution itself"*. Worth the swap when
a distribution has two humps, because a box plot cannot show that at all:

In [ ]:
plt.figure(figsize=(8, 4))
sns.violinplot(data=bikes, x="season", y="cnt", hue="season",
               legend=False, palette="muted")
plt.xticks([0, 1, 2, 3], [seasons[s] for s in [1, 2, 3, 4]])
plt.title("The same four groups, with their shapes")
plt.show()

Every season is fat at the bottom — most hours are quiet hours. A box
plot reports four tidy medians and never tells you that.

### 7.4 Quantiles  *(slide 34)*

Slide 34 explains why quantiles dominate reporting: they create a sense
of what is **normal**. *"90% of calls last less than 3 minutes 22
seconds"* needs no statistics to understand.

In [ ]:
for q in [0.5, 0.8, 0.9, 0.95, 0.99]:
    print("%2d%% of hours had fewer than %4.0f hires"
          % (q * 100, bikes["cnt"].quantile(q)))

Slide 34 then asks a question and answers it, and it is the best question
on the slide:

> **Q: what would a plot of all possible quantiles represent?**
> **A: the cumulative probability function.**

Draw it and it is obvious:

In [ ]:
sorted_counts = np.sort(bikes["cnt"])
share_below = np.arange(1, len(sorted_counts) + 1) / len(sorted_counts)

plt.figure(figsize=(7, 4))
plt.plot(sorted_counts, share_below)
plt.axhline(0.5, linestyle="--", linewidth=1)
plt.axvline(bikes["cnt"].median(), linestyle="--", linewidth=1)
plt.xlabel("hires per hour")
plt.ylabel("share of hours at or below")
plt.title("Every quantile at once — the cumulative distribution")
plt.show()

Sort the values, count upwards, plot. The dashed lines cross at the
median, because the median *is* the 50th percentile. Read any height off
the curve and you have that quantile.

This picture is also where **p-values** come from — the same curve,
Wednesday's session.

### 7.5 Scatterplots  *(slide 30)*

Two columns, one point each:

In [ ]:
plt.figure(figsize=(6.5, 4.5))
plt.scatter(bikes["temp_c"], bikes["cnt"], s=4, alpha=0.15)
plt.xlabel("temperature (C)")
plt.ylabel("hires that hour")
plt.title("17,379 points — with alpha, so density shows")
plt.show()

`alpha=0.15` is doing the work. At full opacity this is a solid black
blob; faded, the *density* becomes visible and you can see where the
points actually pile up. **Always set alpha on a big scatter.**

### 7.6 Pairwise correlations  *(slides 35, 36)*

`df.corr()` computes a correlation between every pair of numeric columns
— **Pearson's by default**, which measures *straight-line* association
only:

In [ ]:
numeric = bikes[["temp_c", "hum_pct", "wind_kmh", "casual", "registered", "cnt"]]
correlations = numeric.corr()
print(correlations.round(2))

Slide 35 notes that only the figures below (or above) the diagonal are
needed — the matrix is symmetric, and the diagonal is 1 by definition.
That is what a heat map with a mask shows *(slide 42)*:

In [ ]:
mask = np.triu(np.ones_like(correlations, dtype=bool))

plt.figure(figsize=(6.5, 5))
sns.heatmap(correlations, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Correlation heat map (upper half hidden — it is a mirror)")
plt.show()

Read it as sentences:

- `cnt` and `registered`: **0.97**. Almost the same column — of course,
  registered riders are most of the total.
- `cnt` and `temp_c`: **0.40**. Warm weather goes with more hires.
- `cnt` and `hum_pct`: **−0.32**. Muggy weather goes with fewer.
- `temp_c` and `casual`: **0.46**, higher than temperature's correlation
  with registered riders (**0.34**). Casual riders are more
  weather-sensitive than commuters — a real finding, from one table.


In [ ]:
sample = bikes.sample(600, random_state=0)

sns.pairplot(sample[["temp_c", "hum_pct", "wind_kmh", "cnt"]],
             plot_kws={"s": 8, "alpha": 0.4})
plt.suptitle("Pair plot (slide 36) — every pair, plus each distribution",
             y=1.01)
plt.show()

Slide 36's `sns.pairplot`. Diagonal: each column's own distribution.
Off-diagonal: every pair as a scatter.

**Sampled to 600 rows on purpose** — a pairplot of 17,379 points draws
sixteen unreadable blobs and takes far longer. That is section 9's
message arriving early.

Now use it. The `cnt` row is the one to read, and it shows something the
correlation number could not: the temperature relationship **bends**.
Demand climbs with warmth and then flattens near the top, because 35°C is
too hot to cycle. Pearson's `r` of 0.40 reports a straight line through a
curve.

> **Checkpoint.** A correlation matrix tells you where to look. A pair plot tells you whether the number meant what you assumed. Run both, in that order.

---

## More than two dimensions  


### Three-dimensional plots

Matplotlib will draw a genuine 3-D scatter. It is usually a worse chart
than the colour-coded 2-D version — depth is ambiguous on a flat screen
and points hide behind each other — but you should know how:

In [ ]:
sample3d = bikes.sample(1200, random_state=2)

figure = plt.figure(figsize=(7, 5.5))
ax = figure.add_subplot(projection="3d")
ax.scatter(sample3d["temp_c"], sample3d["hum_pct"], sample3d["cnt"],
           s=4, alpha=0.4)
ax.set_xlabel("temp (C)")
ax.set_ylabel("humidity (%)")
ax.set_zlabel("hires")
plt.title("Three axes — compare this with the coloured scatter above")
plt.show()

Static, it is hard to read. Rotate it interactively (`%matplotlib widget`)
and it improves a lot. Slide 41's **slicing** is the alternative the deck
offers: hold one variable fixed and view the plane — which is what
`bikes[bikes["hr"] == 8]` has been doing all along.


---

## Large datasets and sampling  *(slides 50, 51)*

Seventeen thousand rows fit comfortably in memory. Seventeen million will
not, and the pair plot above was already slow at full size.

Slide 50's answer is **sampling**, and pandas makes it one call:

In [ ]:
sample = bikes.sample(1000, random_state=42)

print("full mean:   %.1f" % bikes["cnt"].mean())
print("sample mean: %.1f" % sample["cnt"].mean())
print("difference:  %.1f" % abs(bikes["cnt"].mean() - sample["cnt"].mean()))

A thousand rows out of 17,379 — under 6% — lands close. `random_state`
fixes the draw so your notebook gives the same answer twice, which
matters more than it sounds.

But *one* sample is one number, and you cannot see its uncertainty. Slide
50's second idea is **repeated sampling** — take many, and look at how
they scatter:

In [ ]:
rng = np.random.default_rng(0)
means = [bikes["cnt"].sample(10000, random_state=rng.integers(1e6)).mean()
         for _ in range(2000)]

plt.figure(figsize=(8, 3.5))
plt.hist(means, bins=40)
plt.axvline(bikes["cnt"].mean(), color="black", linestyle="--",
            label="true mean")
plt.legend()
plt.title("2,000 sample means, each from 1,000 rows")
plt.xlabel("sample mean hires")
plt.show()

### The central limit theorem  *(slide 51)*

Look at what just happened, because it is the most useful fact in applied
statistics.

`cnt` is **heavily right-skewed** — we measured a skew of 1.28 and drew
the histogram. But the distribution of its **sample means** is a
**bell**, centred on the true mean.

That is slide 51: take samples of size *n*, compute each mean, and as *n*
grows the set of means approaches a normal distribution whose centre is
the population mean. It holds **regardless of the original
distribution's shape**, provided the samples are independent and
identically distributed.

In [ ]:
print("population mean: %.2f" % bikes["cnt"].mean())
print("mean of the 2,000 sample means: %.2f" % np.mean(means))
print()


> **Checkpoint.** Skewed data, bell-shaped sample means, spread exactly s/root-n. That is the CLT — the bridge from the statistics session to every confidence interval you will ever compute.

---

## 11. The checklist

This is the section to keep. Open it at the start of every project,
including Mini Project 1.

**Before you load**

- [ ] Where did this come from, and who owns it?
- [ ] Is there a **data dictionary**? Read it first. (`temp` ÷ 41.)
- [ ] What phenomenon is this a **proxy** for? What can it not see?

**Load**

- [ ] Check `.shape` immediately. One column means the wrong delimiter.
- [ ] Check `.columns` against what you were promised.
- [ ] Check `.head()` **and** `.tail()` — trailing junk rows live at the end.

**Types**

- [ ] `.dtypes` on everything. `object` where you expected a number is a
      problem, not a preference.
- [ ] Dates converted with an explicit format; day-first assumption
      written down.
- [ ] Numbers-that-are-labels identified and mapped to their meanings.

**Missing**

- [ ] `.isnull().sum()` per column.
- [ ] For time series, **reindex onto the full timeline** — absent rows
      are invisible to `isnull`.
- [ ] Decide per column: leave, impute, drop rows, drop column. Write down
      why.
- [ ] If you impute, keep a `_was_missing` flag.

**Invalid**

- [ ] Min and max on every numeric column. Are they physically possible?
- [ ] Sentinel values: 0, −1, 999, 1900-01-01, "N/A", "unknown".
- [ ] Do the invalid rows **cluster** in time or in one group?
- [ ] Assert the relationships the documentation claims.

**Distributions**

- [ ] Histogram of every continuous column. Vary the bins.
- [ ] Skew and kurtosis. Skewed means prefer median and IQR.
- [ ] `value_counts()` on every categorical column, and look at the
      **small** categories.

**Relationships**

- [ ] `.corr()` to find candidates.
- [ ] **Plot every correlation you intend to report.**
- [ ] Stratify by a categorical variable — the relationship may reverse.

**Outliers**

- [ ] Both rules, and note where they disagree.
- [ ] Conditional outliers: unusual *for their group*.
- [ ] For each one: error, or phenomenon? Look the dates up.
- [ ] If unsure, run the analysis with and without, and report both.

**Before you model**

- [ ] Can this data answer the original question? Say so if not.
- [ ] What did you change, and what did that cost?

## Your turn

**1.** Take `casual` instead of `cnt`. Compute skew and kurtosis, draw
the histogram, and decide which outlier rule you would use and why.

**2.** Build a contingency table of `season_name` against `holiday`,
normalised so each row sums to 100. Then say in one sentence what
question that normalisation answers.

**3.** The `windspeed` column has 2,180 rows of exactly 0.0. Investigate
the way section 4.4 did: how many, and do they cluster? Decide whether
they are real calm hours or a sensor floor, and defend your answer.

**4.** Repeat the CLT experiment with samples of 50 rows instead of
1,000. Does the bell still appear? Is the standard error still `s/√n`?

In [ ]:
# 2. Contingency table, row-normalised
answer = pd.crosstab(bikes["season_name"], bikes["holiday"],
                     normalize="index") * 100
print(answer.round(1))

assert np.allclose(answer.sum(axis=1), 100)
print("check passed: every row sums to 100")

In [ ]:
# 3. Starting point — the counting is done for you, the judgement is not
zero_wind = bikes[bikes["windspeed"] == 0]
print("rows with windspeed exactly 0:", len(zero_wind))
print("distinct dates involved:", zero_wind["dteday"].nunique())
print("distinct dates in the file:", bikes["dteday"].nunique())

## Stretch

**A.** `registered` correlates with `cnt` at 0.97. Explain in one
sentence why including both in a model would be a problem, and name the
concept. (It arrives properly in Module 4.)

**B.** Slide 24 mentions proximity-based outlier detection. Implement the
simplest version: standardise `temp_c`, `hum_pct` and `wind_kmh`, compute
each day's Euclidean distance to the mean of all days, and list the five
furthest. Are they the same days section 6.2 found?

**C.** Rebuild the hour-by-weekday heat map for `casual` riders only, and
put it beside the `registered` one. Describe the difference in two
sentences.

## Where to go deeper

- **Python for Data Analysis**, Wes McKinney (3rd ed.) —
  <https://wesmckinney.com/book/>: free online, written by the author of
  pandas. Chapters 7 and 8 are data cleaning and preparation.
- **Fundamentals of Data Visualization**, Claus Wilke —
  <https://clauswilke.com/dataviz/>: free online, and the best short
  explanation of which chart to use and why.
- **seaborn tutorial** —
  <https://seaborn.pydata.org/tutorial.html>: the distribution and
  categorical sections cover most of sections 7 and 8 above.
- **Seeing Theory** (Brown University) —
  <https://seeing-theory.brown.edu>: the *Probability Distributions*
  chapter lets you drag a distribution around and watch the CLT happen.

---

*Data Science & AI — Module 2 Part 1, slides 3-51. Reference notebook,
not a session. Official labs: IOD Lab 2.1.1 Data Wrangling and Munging,
IOD Lab 2.1.2 Data Profiling.*